#### Импорт

In [6]:
import os
import sys
import math
import itertools
from pathlib import Path

# Добавляем путь на уровень выше
sys.path.append(str(Path(os.getcwd()).resolve().parent))
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


from utils.features import *
from utils.load_data import load_all_data

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.utils.tensorboard import SummaryWriter
from torchinfo import summary
from tqdm import tqdm

#### Загрузка данных

In [2]:
data_dir = Path('../data/PEMS03')
metadata, data, adj = load_all_data(data_dir)
data = data[:2016]
data = data.copy()

data[:, :, 1] = data[:, :, 1] * 288
data[:, :, 2] = data[:, :, 2] * 7

print(f"data.shape: {data.shape}")

data.shape: (2016, 358, 3)


#### DataLoader

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Данные и параметры
L, N, C = data.shape  # [2016, N, C]
batch_size = 16
train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2
seq_len = 12  # Количество временных шагов на вход
pred_len = 12  # Количество временных шагов для предсказания
num_lags = 12

# Индексы каналов для нормализации


# Проверка корректности разделения данных
assert train_ratio + val_ratio + test_ratio == 1.0, "Сумма долей train, val и test должна быть равна 1.0"

# Разделение данных на train, val и test
num_samples = data.shape[0]  # Количество временных шагов (L)
train_size = int(num_samples * train_ratio)
val_size = int(num_samples * val_ratio)
test_size = num_samples - train_size - val_size

train_data = data[:train_size, :, :]  # [train_size, N, C]
val_data = data[train_size:train_size + val_size, :, :]  # [val_size, N, C]
test_data = data[train_size + val_size:, :, :]  # [test_size, N, C]

def normalize_data(train_data, val_data, test_data, channels_to_normalize, eps=1e-5):
    assert all(0 <= ch < train_data.shape[2] for ch in channels_to_normalize), "Недопустимые индексы каналов"

    # Вычисляем среднее и стандартное отклонение по обучающим данным
    mean = train_data[:, :, channels_to_normalize].mean(axis=(0, 1), keepdims=True)  # [1, 1, C_norm]
    std = train_data[:, :, channels_to_normalize].std(axis=(0, 1), keepdims=True)    # [1, 1, C_norm]
    std[std < eps] = 1.0  # защита от деления на 0

    # Нормализация
    train_data[:, :, channels_to_normalize] = (train_data[:, :, channels_to_normalize] - mean) / std
    val_data[:, :, channels_to_normalize] = (val_data[:, :, channels_to_normalize] - mean) / std
    test_data[:, :, channels_to_normalize] = (test_data[:, :, channels_to_normalize] - mean) / std

    return train_data, val_data, test_data, mean, std

# Нормализация данных
normalize = True
if normalize:
    channels_to_normalize = [0]  # например, только поток
    train_data, val_data, test_data, mean, std = normalize_data(train_data, val_data, test_data, channels_to_normalize)


# Создание кастомного Dataset
class TrafficDataset(Dataset):
    def __init__(self, data, seq_len, pred_len, num_lags):
        """
        Параметры:
          data: тензор формы [L, N, C]
          seq_len: длина входной последовательности
          pred_len: длина предсказываемой последовательности
          num_lags: число лагов, добавляемых к признакам (из канала C=0)
        """
        super().__init__()
        self.data = data  # [L, N, C]
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.num_lags = num_lags
        

    def __len__(self):
        # Количество возможных последовательностей
        return self.data.shape[0] - self.seq_len - self.pred_len + 1

    def __getitem__(self, idx):
        # Извлекаем основную последовательность входных данных
        x_main = self.data[idx: idx + self.seq_len, :, :]  # [seq_len, N, C]

        # Формируем матрицу лагов для канала C=0 внутри текущей последовательности.
        # Если для временного шага t нет достаточного количества предыдущих значений,
        # соответствующие позиции заполняются нулями.
        lag_matrix = torch.zeros((self.seq_len, self.data.shape[1], self.num_lags), device=self.data.device)
        for t in range(self.seq_len):
            # Для каждого лага, lag=1 соответствует непосредственному предыдущему значению
            for lag in range(1, self.num_lags + 1):
                if t - lag >= 0:
                    lag_matrix[t, :, lag - 1] = x_main[t - lag, :, 0]
                # Если t - lag < 0, оставляем нули

        # Объединяем исходные признаки и лаги по последнему измерению
        x = torch.cat([x_main, lag_matrix], dim=-1)  # [seq_len, N, C + num_lags]

        # Целевые значения – поток (канал C=0) для последовательности предсказания
        y = self.data[idx + self.seq_len: idx + self.seq_len + self.pred_len, :, 0]  # [pred_len, N]

        return x, y

# Преобразование данных в тензоры с dtype=torch.float32
train_data = torch.tensor(train_data, dtype=torch.float32).to(device)
val_data = torch.tensor(val_data, dtype=torch.float32).to(device)
test_data = torch.tensor(test_data, dtype=torch.float32).to(device)

# Создание DataLoader
train_dataset = TrafficDataset(train_data, seq_len, pred_len, num_lags)
val_dataset = TrafficDataset(val_data, seq_len, pred_len, num_lags)
test_dataset = TrafficDataset(test_data, seq_len, pred_len, num_lags)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Using device: cuda


In [4]:
for x, y in train_loader:
    print(f"Shape of x (input data):    {x.shape}")  # [B, L, N, C]
    print(f"Shape of y (target data):   {y.shape}")  # [B, L, N]

    # Проверка первых двух сенсоров и временных шагов
    print("\nFirst two sensors and time steps in x:")
    print(x[0, :2, :2, :3])  # Первый батч, первые два временных шага, первые два сенсора, все каналы

    print("\nFirst two sensors and time steps in y:")
    print(y[0, :2, :2])  # Первый батч, первые два временных шага, первые два сенсора

    print("\nExample lags data:")
    print(x[0, :, 0, [3,4,5,6,7,8,9,10,11,12,13,14]])

    # Проверка типов данных для всех каналов
    print("\nData types for each channel in x:")
    for channel in range(x.shape[3]):  # Проходим по всем каналам
        print(f"Channel {channel} dtype: {x[0, 0, 0, channel].dtype}, device: {x[0, 0, 0, channel].device}")

    # Остановка выполнения для ручной проверки
    break

Shape of x (input data):    torch.Size([16, 12, 358, 15])
Shape of y (target data):   torch.Size([16, 12, 358])

First two sensors and time steps in x:
tensor([[[ -0.3372, 121.0000,   0.0000],
         [ -0.3809, 121.0000,   0.0000]],

        [[ -0.1477, 122.0000,   0.0000],
         [ -0.1549, 122.0000,   0.0000]]], device='cuda:0')

First two sensors and time steps in y:
tensor([[-0.1258, -0.1258],
        [-0.0383, -0.1185]], device='cuda:0')

Example lags data:
tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [-0.3372,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [-0.1477, -0.3372,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [-0.1185, -0.1477, -0.3372,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0564, -0.

#### GWNet Model

In [ ]:
# 1. LearnableFilter: фильтрация входного сигнала с помощью FFT
class LearnableFilter(nn.Module):
    def __init__(
        self,
        num_features,
        seq_len,
        init_range=0.0002,
        freq_mask_threshold=None  # например, оставить первые 20 частот
    ):
        """
        num_features: число каналов для фильтрации
        seq_len: длина входной последовательности (L)
        init_range: диапазон инициализации весов фильтра (для стабильности)
        freq_mask_threshold: если не None, используем только фильтры по низким частотам (например, 20)
        """
        super().__init__()
        self.seq_len = seq_len
        self.num_features = num_features
        # Размерность FFT-части (только положительные частоты)
        self.fft_len_half = seq_len // 2 + 1

        real_part = torch.zeros(1, num_features, 1, self.fft_len_half)
        imag_part = torch.zeros(1, num_features, 1, self.fft_len_half)
        nn.init.uniform_(real_part, -init_range, init_range)
        nn.init.uniform_(imag_part, -init_range, init_range)

        self.K_real = nn.Parameter(real_part)
        self.K_imag = nn.Parameter(imag_part)

        if freq_mask_threshold is not None:
            self.freq_mask_threshold = freq_mask_threshold
        else:
            self.freq_mask_threshold = self.fft_len_half // 2

    def forward(self, x):
        """
        x: (B, Channels, N, L)
        """
        batch, channels, nodes, seq_len_in = x.shape
        assert seq_len_in == self.seq_len, f"Входной seq_len={seq_len_in}, ожидалось {self.seq_len}"

        x_fft = torch.fft.rfft(x, n=self.seq_len, dim=-1)  # (B, Channels, N, fft_len_half)

        # (1) Маска частот, если задан порог (например, freq_mask_threshold = 20)
        K_real = self.K_real
        K_imag = self.K_imag
        if self.freq_mask_threshold is not None:
            mask = torch.zeros_like(K_real)
            mask[..., :self.freq_mask_threshold] = 1.0
            K_real = K_real * mask
            K_imag = K_imag * mask

        K = torch.complex(K_real, K_imag).to(x.device)  # Преобразуем в комплексное число

        # (2) Применение фильтра
        x_filtered_fft = x_fft * K  # Broadcasting

        # (3) Обратное FFT
        x_filtered = torch.fft.irfft(x_filtered_fft, n=self.seq_len, dim=-1)
        if torch.isnan(x_filtered).any():
            raise RuntimeError("NaN найден в результате LearnableFilter")
        return x_filtered


# 2. TimeEmbedding: получение эмбеддингов времени с дополнением последовательности при необходимости
class TimeEmbedding(nn.Module):
    def __init__(self, num_embeddings=288, emb_dim=4):
        """
        Args:
            num_embeddings: количество уникальных временных индексов.
            emb_dim: размерность эмбеддингов времени.
        """
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, emb_dim)

    def forward(self, time_idx: torch.Tensor, target_seq_len: int):
        """
        Args:
            time_idx: тензор индексов времени формы (B, N, L).
            target_seq_len: требуемая длина последовательности.
        Returns:
            Эмбеддинги времени формы (B, emb_dim, N, L)
        """
        seq_len = time_idx.shape[-1]
        if seq_len < target_seq_len:
            time_idx = F.pad(time_idx, (target_seq_len - seq_len, 0), "constant", 0)
        time_emb = self.embedding(time_idx)
        # Перестановка размерностей для конкатенации: (B, N, L, emb_dim) -> (B, emb_dim, N, L)
        time_emb = time_emb.permute(0, 3, 1, 2)
        return time_emb

# 3. LagsExtractor: выделение дополнительных каналов (лагов) с дополнением длины последовательности
class LagsExtractor(nn.Module):
    def __init__(self, receptive_field, start_channel=3):
        """
        Args:
            receptive_field: минимальная длина последовательности (требуемый receptive field).
            start_channel: индекс, с которого начинаются лаги.
        """
        super().__init__()
        self.receptive_field = receptive_field
        self.start_channel = start_channel

    def forward(self, input_tensor: torch.Tensor):
        """
        Args:
            input_tensor: входной тензор формы (B, C, N, L)
        Returns:
            Лаги или None, если их нет.
        """
        if input_tensor.shape[1] > self.start_channel:
            lags = input_tensor[:, self.start_channel:, :, :]
            seq_len = lags.size(3)
            if seq_len < self.receptive_field:
                lags = F.pad(lags, (self.receptive_field - seq_len, 0, 0, 0))
            return lags
        return None

# 4. AdjacencyMatrixGenerator: динамическая генерация матрицы смежности
class AdjacencyMatrixGenerator(nn.Module):
    def __init__(self, num_nodes):
        """
        Args:
            num_nodes: число узлов графа.
        """
        super().__init__()
        self.num_nodes = num_nodes
        self.W = nn.Parameter(torch.randn(num_nodes, num_nodes))

    def forward(self, X: torch.Tensor):
        """
        Args:
            X: входной тензор формы (B, C, N, L)
        Returns:
            Динамическую матрицу смежности формы (B, N, N)
        """
        # Перестановка: (B, C, N, L) -> (B, N, L, C)
        X_perm = X.permute(0, 2, 3, 1)
        # Используем часть каналов (начиная с 3-го)
        data_window = X_perm[:, :, 3:, :]
        data_centered = data_window - data_window.mean(dim=2, keepdim=True)
        fft_features = torch.fft.rfft(data_centered, dim=2).real
        fft_features = fft_features.mean(dim=-1)  # Среднее по последнему измерению

        cov = torch.matmul(fft_features, fft_features.transpose(1, 2))
        std = torch.sqrt(torch.sum(fft_features ** 2, dim=2))
        std_safe = torch.where(std == 0, torch.full_like(std, 1e-8), std)
        corr = cov / (std_safe.unsqueeze(1) * std_safe.unsqueeze(2) + 1e-8)

        A_dynamic = torch.matmul(torch.matmul(corr, self.W), corr.transpose(1, 2))
        A_dynamic = F.softmax(F.relu(A_dynamic), dim=-1)
        return A_dynamic

# Дополнительные компоненты для графовой свёрточной части (оставлены без изменений)
class nconv(nn.Module):
    """Операция графовой свёртки."""
    def __init__(self):
        super().__init__()

    def forward(self, x, A):
        if len(A.shape) == 2:
            x = torch.einsum('ncvl,vw->ncwl', (x, A))
        elif len(A.shape) == 3:
            x = torch.einsum('ncvl,nvw->ncwl', (x, A))
        return x.contiguous()

class linear(nn.Module):
    """Линейный слой (1x1 свёртка)."""
    def __init__(self, c_in, c_out):
        super().__init__()
        self.mlp = nn.Conv2d(c_in, c_out, kernel_size=(1, 1), bias=True)

    def forward(self, x):
        return self.mlp(x)

class gcn(nn.Module):
    """Графовая свёрточная сеть."""
    def __init__(self, c_in, c_out, dropout, support_len=3, order=2):
        super().__init__()
        self.nconv = nconv()
        c_in_total = (order * support_len + 1) * c_in
        self.mlp = linear(c_in_total, c_out)
        self.dropout = dropout
        self.order = order

    def forward(self, x, support):
        out = [x]
        for a in support:
            x1 = self.nconv(x, a.to(x.device))
            out.append(x1)
            for k in range(2, self.order + 1):
                x1 = self.nconv(x1, a.to(x.device))
                out.append(x1)
        h = torch.cat(out, dim=1)
        h = self.mlp(h)
        h = F.dropout(h, self.dropout, training=self.training)
        return h

# 5. Основная модель GWNet с возможностью включения/отключения отдельных компонентов
class GWNet(nn.Module):
    def __init__(self,
                 num_nodes,
                 dropout=0.3,
                 supports=None,
                 gcn_bool=True,
                 addaptadj=True,
                 aptinit=None,
                 in_dim=2,
                 out_dim=12,
                 residual_channels=32,
                 dilation_channels=32,
                 skip_channels=256,
                 end_channels=512,
                 kernel_size=2,
                 blocks=4,
                 layers=2,
                 emb_dim=4,
                 add_c=0,
                 use_filter=False,
                 use_time_emb=False,
                 use_lags=False,
                 use_dynamic_adj=False,
                 filter_init_range=0.002):
        """
        Args:
            num_nodes: число узлов.
            dropout: коэффициент dropout.
            supports: список матриц смежности, если есть.
            gcn_bool: флаг использования GCN.
            addaptadj: флаг использования адаптивной смежности.
            aptinit: инициализация для адаптивной смежности.
            in_dim: число входных каналов.
            out_dim: длина выходной последовательности.
            residual_channels, dilation_channels, skip_channels, end_channels: размеры каналов.
            kernel_size: размер ядра для TCN.
            blocks, layers: структура TCN.
            emb_dim: размерность эмбеддингов времени/узлов.
            add_c: дополнительные каналы.
            use_filter: применять ли LearnableFilter.
            use_time_emb: применять ли TimeEmbedding.
            use_lags: использовать ли выделение lag-показателей.
            use_dynamic_adj: использовать ли динамическую матрицу смежности.
            filter_init_range: диапазон инициализации для фильтра.
        """
        super().__init__()
        self.use_filter = use_filter
        self.use_time_emb = use_time_emb
        self.use_lags = use_lags
        self.use_dynamic_adj = use_dynamic_adj
        self.emb_dim = emb_dim
        self.dropout = dropout
        self.blocks = blocks
        self.layers = layers
        self.gcn_bool = gcn_bool
        self.addaptadj = addaptadj
        self.window_size = 12  # можно сделать параметром, если потребуется

        # Инициализация time и node эмбеддингов
        if self.use_time_emb:
            self.time_embedding = TimeEmbedding(num_embeddings=288, emb_dim=emb_dim)
        self.node_embedding = nn.Embedding(num_nodes, emb_dim)

        # Начальный свёрточный слой
        # Если используется TimeEmbedding и/или лаги, входное число каналов увеличивается
        extra_channels = 0
        if self.use_time_emb:
            extra_channels += emb_dim
        if self.use_lags:
            # Полагем, что число лаговых каналов определяется динамически (из оставшихся)
            # В базовом случае lags будут содержаться, если in_dim > 2
            extra_channels += 12
        self.start_conv = nn.Conv2d(in_channels=in_dim + extra_channels + add_c,
                                    out_channels=residual_channels,
                                    kernel_size=(1, 1))

        # Опциональный фильтр
        if self.use_filter:
            self.filter = LearnableFilter(num_features=1, seq_len=out_dim, init_range=filter_init_range)
        else:
            self.filter = None

        self.supports = supports
        self.supports_len = 0
        if supports is not None:
            self.supports_len += len(supports)
        if gcn_bool and addaptadj:
            if aptinit is None:
                self.nodevec1 = nn.Parameter(torch.randn(num_nodes, 10))
                self.nodevec2 = nn.Parameter(torch.randn(10, num_nodes))
                self.supports_len += 1
                if self.use_dynamic_adj:
                    self.supports_len += 1
            else:
                if supports is None:
                    supports = []
                m, p, n = torch.svd(aptinit)
                initemb1 = torch.mm(m[:, :10], torch.diag(p[:10] ** 0.5))
                initemb2 = torch.mm(torch.diag(p[:10] ** 0.5), n[:, :10].t())
                self.nodevec1 = nn.Parameter(initemb1)
                self.nodevec2 = nn.Parameter(initemb2)
                self.supports_len += 1
                if self.use_dynamic_adj:
                    self.supports_len += 1

        # Инициализация модулей TCN и GCN
        self.filter_convs = nn.ModuleList()
        self.gate_convs = nn.ModuleList()
        self.residual_convs = nn.ModuleList()
        self.skip_convs = nn.ModuleList()
        self.bn = nn.ModuleList()
        self.gconv = nn.ModuleList()

        receptive_field = 1

        for b in range(blocks):
            additional_scope = kernel_size - 1
            new_dilation = 1
            for i in range(layers):
                self.filter_convs.append(nn.Conv2d(in_channels=residual_channels,
                                                   out_channels=dilation_channels,
                                                   kernel_size=(1, kernel_size),
                                                   dilation=new_dilation))
                self.gate_convs.append(nn.Conv2d(in_channels=residual_channels,
                                                 out_channels=dilation_channels,
                                                 kernel_size=(1, kernel_size),
                                                 dilation=new_dilation))
                self.residual_convs.append(nn.Conv2d(in_channels=dilation_channels,
                                                     out_channels=residual_channels,
                                                     kernel_size=(1, 1)))
                self.skip_convs.append(nn.Conv2d(in_channels=dilation_channels,
                                                 out_channels=skip_channels,
                                                 kernel_size=(1, 1)))
                self.bn.append(nn.BatchNorm2d(residual_channels))
                new_dilation *= 2
                receptive_field += additional_scope
                additional_scope *= 2
                if self.gcn_bool:
                    self.gconv.append(gcn(dilation_channels, residual_channels, dropout, support_len=self.supports_len))
        self.receptive_field = receptive_field

        self.end_conv_1 = nn.Conv2d(in_channels=skip_channels,
                                    out_channels=end_channels,
                                    kernel_size=(1, 1))
        self.end_conv_2 = nn.Conv2d(in_channels=end_channels,
                                    out_channels=out_dim,
                                    kernel_size=(1, 1))

        # Опциональный модуль для выделения лагов
        if self.use_lags:
            self.lags_extractor = LagsExtractor(receptive_field=self.receptive_field, start_channel=3)
        else:
            self.lags_extractor = None

        # Опциональный модуль генерации динамической матрицы смежности
        if self.use_dynamic_adj:
            self.adj_generator = AdjacencyMatrixGenerator(num_nodes)
        else:
            self.adj_generator = None

    def compute_adaptive_supports(self, input_tensor):
        new_supports = None
        if self.gcn_bool and self.addaptadj and self.supports is not None:
            adp1 = F.softmax(F.relu(torch.mm(self.nodevec1, self.nodevec2)), dim=1)
            new_supports = self.supports + [adp1]
            if self.use_dynamic_adj:
                adp2 = self.adj_generator(input_tensor)
                new_supports.append(adp2)
        return new_supports

    def forward(self, history_data: torch.Tensor) -> torch.Tensor:
        """
        Args:
            history_data: входной тензор формы (B, L, N, C)
        Returns:
            Выходной тензор формы (B, out_dim, N, 1)
        """
        # Транспонируем в форму (B, C, N, L)
        x = history_data.transpose(1, 3).contiguous()

        # Извлечение основных каналов: скорость (0-й канал) и время (1-й канал)
        speed = x[:, [0], :, :]  # форма: (B, 1, N, L)
        time = x[:, 1, :, :]     # форма: (B, N, L)

        # Применение фильтра к скорости, если требуется
        if self.use_filter:
            speed_filtered = self.filter(speed)
        else:
            speed_filtered = speed

        in_len = speed_filtered.size(3)
        # Дополнение последовательности до требуемой длины (receptive_field) при необходимости
        if in_len < self.receptive_field:
            speed_filtered = F.pad(speed_filtered, (self.receptive_field - in_len, 0, 0, 0))
            speed = F.pad(speed, (self.receptive_field - in_len, 0, 0, 0))

        # Объединение каналов: скорость, время и лаги (если есть)
        components = [speed_filtered]

        if self.use_time_emb:
            time_emb = self.time_embedding(time.long(), target_seq_len=self.receptive_field)
            components.append(time_emb)

        if self.use_lags:
            lags = self.lags_extractor(x)
            if lags is not None:
                components.append(lags)

        x_combined = torch.cat(components, dim=1)
        x_combined = self.start_conv(x_combined)

        skip = 0
        new_supports = self.compute_adaptive_supports(x)

        # TCN + GCN блоки
        for i in range(self.blocks * self.layers):
            residual = x_combined
            filter_out = torch.tanh(self.filter_convs[i](residual))
            gate_out = torch.sigmoid(self.gate_convs[i](residual))
            x_combined = filter_out * gate_out

            s = self.skip_convs[i](x_combined)
            if isinstance(skip, torch.Tensor):
                # Усечение skip, если требуется
                skip = skip[:, :, :, -s.size(3):]
            else:
                skip = 0
            skip = s + skip

            # Применяем либо GCN, либо обычную свёрточную сеть для остаточного соединения
            if self.gcn_bool and self.supports is not None:
                if self.addaptadj:
                    x_combined = self.gconv[i](x_combined, new_supports)
                else:
                    x_combined = self.gconv[i](x_combined, self.supports)
            else:
                x_combined = self.residual_convs[i](x_combined)

            # Остаточное соединение с усечением по длине
            x_combined = x_combined + residual[:, :, :, -x_combined.size(3):]
            x_combined = self.bn[i](x_combined)

        out = F.relu(skip)
        out = F.relu(self.end_conv_1(out))
        out = self.end_conv_2(out)
        return out


#### STGCN

In [ ]:
class Align(nn.Module):
    def __init__(self, c_in, c_out):
        super(Align, self).__init__()
        self.c_in = c_in
        self.c_out = c_out
        self.align_conv = nn.Conv2d(
            in_channels=c_in, out_channels=c_out, kernel_size=(1, 1))

    def forward(self, x):
        if self.c_in > self.c_out:
            x = self.align_conv(x)
        elif self.c_in < self.c_out:
            batch_size, _, timestep, n_vertex = x.shape
            x = torch.cat([x, torch.zeros(
                [batch_size, self.c_out - self.c_in, timestep, n_vertex]).to(x)], dim=1)
        else:
            x = x

        return x


class CausalConv1d(nn.Conv1d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, enable_padding=False, dilation=1, groups=1, bias=True):
        if enable_padding == True:
            self.__padding = (kernel_size - 1) * dilation
        else:
            self.__padding = 0
        super(CausalConv1d, self).__init__(in_channels, out_channels, kernel_size=kernel_size,
                                           stride=stride, padding=self.__padding, dilation=dilation, groups=groups, bias=bias)

    def forward(self, input):
        result = super(CausalConv1d, self).forward(input)
        if self.__padding != 0:
            return result[:, :, : -self.__padding]

        return result


class CausalConv2d(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, enable_padding=False, dilation=1, groups=1, bias=True):
        kernel_size = nn.modules.utils._pair(kernel_size)
        stride = nn.modules.utils._pair(stride)
        dilation = nn.modules.utils._pair(dilation)
        if enable_padding == True:
            self.__padding = [int((kernel_size[i] - 1) * dilation[i])
                              for i in range(len(kernel_size))]
        else:
            self.__padding = 0
        self.left_padding = nn.modules.utils._pair(self.__padding)
        super(CausalConv2d, self).__init__(in_channels, out_channels, kernel_size,
                                           stride=stride, padding=0, dilation=dilation, groups=groups, bias=bias)

    def forward(self, input):
        if self.__padding != 0:
            input = F.pad(
                input, (self.left_padding[1], 0, self.left_padding[0], 0))
        result = super(CausalConv2d, self).forward(input)

        return result


class TemporalConvLayer(nn.Module):

    # Temporal Convolution Layer (GLU)
    #
    #        |--------------------------------| * Residual Connection *
    #        |                                |
    #        |    |--->--- CasualConv2d ----- + -------|
    # -------|----|                                   ⊙ ------>
    #             |--->--- CasualConv2d --- Sigmoid ---|
    #

    # param x: tensor, [bs, c_in, ts, n_vertex]

    def __init__(self, Kt, c_in, c_out, n_vertex, act_func):
        super(TemporalConvLayer, self).__init__()
        self.Kt = Kt
        self.c_in = c_in
        self.c_out = c_out
        self.n_vertex = n_vertex
        self.align = Align(c_in, c_out)
        if act_func == 'glu' or act_func == 'gtu':
            self.causal_conv = CausalConv2d(
                in_channels=c_in, out_channels=2 * c_out, kernel_size=(Kt, 1), enable_padding=False, dilation=1)
        else:
            self.causal_conv = CausalConv2d(in_channels=c_in, out_channels=c_out, kernel_size=(
                Kt, 1), enable_padding=False, dilation=1)
        self.act_func = act_func
        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU()
        self.silu = nn.SiLU()

    def forward(self, x):
        x_in = self.align(x)[:, :, self.Kt - 1:, :]
        x_causal_conv = self.causal_conv(x)

        if self.act_func == 'glu' or self.act_func == 'gtu':
            x_p = x_causal_conv[:, : self.c_out, :, :]
            x_q = x_causal_conv[:, -self.c_out:, :, :]

            if self.act_func == 'glu':
                # GLU was first purposed in
                # *Language Modeling with Gated Convolutional Networks*.
                # URL: https://arxiv.org/abs/1612.08083
                # Input tensor X is split by a certain dimension into tensor X_a and X_b.
                # In the original paper, GLU is defined as Linear(X_a) ⊙ Sigmoid(Linear(X_b)).
                # However, in PyTorch, GLU is defined as X_a ⊙ Sigmoid(X_b).
                # URL: https://pytorch.org/docs/master/nn.functional.html#torch.nn.functional.glu
                # Because in original paper, the representation of GLU and GTU is ambiguous.
                # So, it is arguable which one version is correct.

                # (x_p + x_in) ⊙ Sigmoid(x_q)
                x = torch.mul((x_p + x_in), self.sigmoid(x_q))

            else:
                # Tanh(x_p + x_in) ⊙ Sigmoid(x_q)
                x = torch.mul(self.tanh(x_p + x_in), self.sigmoid(x_q))

        elif self.act_func == 'relu':
            x = self.relu(x_causal_conv + x_in)

        elif self.act_func == 'leaky_relu':
            x = self.leaky_relu(x_causal_conv + x_in)

        elif self.act_func == 'silu':
            x = self.silu(x_causal_conv + x_in)

        else:
            raise NotImplementedError(
                f'ERROR: The activation function {self.act_func} is not implemented.')

        return x


class ChebGraphConv(nn.Module):
    def __init__(self, c_in, c_out, Ks, gso, bias):
        super(ChebGraphConv, self).__init__()
        self.c_in = c_in
        self.c_out = c_out
        self.Ks = Ks
        self.gso = gso
        self.weight = nn.Parameter(torch.FloatTensor(Ks, c_in, c_out))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(c_out))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        #bs, c_in, ts, n_vertex = x.shape
        x = torch.permute(x, (0, 2, 3, 1))

        self.gso = self.gso.to(x.device)

        if self.Ks - 1 < 0:
            raise ValueError(
                f'ERROR: the graph convolution kernel size Ks has to be a positive integer, but received {self.Ks}.')
        elif self.Ks - 1 == 0:
            x_0 = x
            x_list = [x_0]
        elif self.Ks - 1 == 1:
            x_0 = x
            x_1 = torch.einsum('hi,btij->bthj', self.gso, x)
            x_list = [x_0, x_1]
        elif self.Ks - 1 >= 2:
            x_0 = x
            x_1 = torch.einsum('hi,btij->bthj', self.gso, x)
            x_list = [x_0, x_1]
            for k in range(2, self.Ks):
                x_list.append(torch.einsum('hi,btij->bthj', 2 *
                              self.gso, x_list[k - 1]) - x_list[k - 2])

        x = torch.stack(x_list, dim=2)

        cheb_graph_conv = torch.einsum('btkhi,kij->bthj', x, self.weight)

        if self.bias is not None:
            cheb_graph_conv = torch.add(cheb_graph_conv, self.bias)
        else:
            cheb_graph_conv = cheb_graph_conv

        return cheb_graph_conv


class GraphConv(nn.Module):
    def __init__(self, c_in, c_out, gso, bias):
        super(GraphConv, self).__init__()
        self.c_in = c_in
        self.c_out = c_out
        self.gso = gso
        self.weight = nn.Parameter(torch.FloatTensor(c_in, c_out))
        if bias:
            self.bias = nn.Parameter(torch.FloatTensor(c_out))
        else:
            self.register_parameter('bias', None)
        self.reset_parameters()

    def reset_parameters(self):
        init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if self.bias is not None:
            fan_in, _ = init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        #bs, c_in, ts, n_vertex = x.shape
        x = torch.permute(x, (0, 2, 3, 1))

        self.gso = self.gso.to(x.device)

        first_mul = torch.einsum('hi,btij->bthj', self.gso, x)
        second_mul = torch.einsum('bthi,ij->bthj', first_mul, self.weight)

        if self.bias is not None:
            graph_conv = torch.add(second_mul, self.bias)
        else:
            graph_conv = second_mul

        return graph_conv


class GraphConvLayer(nn.Module):
    def __init__(self, graph_conv_type, c_in, c_out, Ks, gso, bias):
        super(GraphConvLayer, self).__init__()
        self.graph_conv_type = graph_conv_type
        self.c_in = c_in
        self.c_out = c_out
        self.align = Align(c_in, c_out)
        self.Ks = Ks
        self.gso = gso
        if self.graph_conv_type == 'cheb_graph_conv':
            self.cheb_graph_conv = ChebGraphConv(c_out, c_out, Ks, gso, bias)
        elif self.graph_conv_type == 'graph_conv':
            self.graph_conv = GraphConv(c_out, c_out, gso, bias)

    def forward(self, x):
        x_gc_in = self.align(x)
        if self.graph_conv_type == 'cheb_graph_conv':
            x_gc = self.cheb_graph_conv(x_gc_in)
        elif self.graph_conv_type == 'graph_conv':
            x_gc = self.graph_conv(x_gc_in)
        x_gc = x_gc.permute(0, 3, 1, 2)
        x_gc_out = torch.add(x_gc, x_gc_in)

        return x_gc_out


class STConvBlock(nn.Module):
    # STConv Block contains 'TGTND' structure
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # G: Graph Convolution Layer (ChebGraphConv or GraphConv)
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # D: Dropout

    def __init__(self, Kt, Ks, n_vertex, last_block_channel, channels, act_func, graph_conv_type, gso, bias, droprate):
        super(STConvBlock, self).__init__()
        self.tmp_conv1 = TemporalConvLayer(
            Kt, last_block_channel, channels[0], n_vertex, act_func)
        self.graph_conv = GraphConvLayer(
            graph_conv_type, channels[0], channels[1], Ks, gso, bias)
        self.tmp_conv2 = TemporalConvLayer(
            Kt, channels[1], channels[2], n_vertex, act_func)
        self.tc2_ln = nn.LayerNorm([n_vertex, channels[2]])
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=droprate)

    def forward(self, x):
        x = self.tmp_conv1(x)
        x = self.graph_conv(x)
        x = self.relu(x)
        x = self.tmp_conv2(x)
        x = self.tc2_ln(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        x = self.dropout(x)

        return x


class OutputBlock(nn.Module):
    # Output block contains 'TNFF' structure
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # F: Fully-Connected Layer
    # F: Fully-Connected Layer

    def __init__(self, Ko, last_block_channel, channels, end_channel, n_vertex, act_func, bias, droprate):
        super(OutputBlock, self).__init__()
        self.tmp_conv1 = TemporalConvLayer(
            Ko, last_block_channel, channels[0], n_vertex, act_func)
        self.fc1 = nn.Linear(
            in_features=channels[0], out_features=channels[1], bias=bias)
        self.fc2 = nn.Linear(
            in_features=channels[1], out_features=end_channel, bias=bias)
        self.tc1_ln = nn.LayerNorm([n_vertex, channels[0]])
        self.relu = nn.ReLU()
        self.leaky_relu = nn.LeakyReLU()
        self.silu = nn.SiLU()
        self.dropout = nn.Dropout(p=droprate)

    def forward(self, x):
        x = self.tmp_conv1(x)
        x = self.tc1_ln(x.permute(0, 2, 3, 1))
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x).permute(0, 3, 1, 2)

        return x


class STGCNChebGraphConv(nn.Module):
    """
    Paper: Spatio-Temporal Graph Convolutional Networks: A Deep Learning Framework for Trafﬁc Forecasting
    Official Code: https://github.com/VeritasYin/STGCN_IJCAI-18 (tensorflow)
    Ref Code: https://github.com/hazdzz/STGCN
    Venue: IJCAI 2018
    Task: Spatial-Temporal Forecasting
    Note:  
        https://github.com/hazdzz/STGCN/issues/9
    Link: https://arxiv.org/abs/1709.04875
    """

    # STGCNChebGraphConv contains 'TGTND TGTND TNFF' structure
    # ChebGraphConv is the graph convolution from ChebyNet.
    # Using the Chebyshev polynomials of the first kind as a graph filter.

    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # G: Graph Convolution Layer (ChebGraphConv)
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # D: Dropout

    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # G: Graph Convolution Layer (ChebGraphConv)
    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normolization
    # D: Dropout

    # T: Gated Temporal Convolution Layer (GLU or GTU)
    # N: Layer Normalization
    # F: Fully-Connected Layer
    # F: Fully-Connected Layer

    def __init__(self, Kt, Ks, blocks, T, n_vertex, act_func, graph_conv_type, gso, bias, droprate):
        super(STGCNChebGraphConv, self).__init__()
        modules = []
        for l in range(len(blocks) - 3):
            modules.append(STConvBlock(
                Kt, Ks, n_vertex, blocks[l][-1], blocks[l+1], act_func, graph_conv_type, gso, bias, droprate))
        self.st_blocks = nn.Sequential(*modules)
        Ko = T - (len(blocks) - 3) * 2 * (Kt - 1)
        self.Ko = Ko
        assert Ko != 0, "Ko = 0."
        self.output = OutputBlock(
            Ko, blocks[-3][-1], blocks[-2], blocks[-1][0], n_vertex, act_func, bias, droprate)

    def forward(self, history_data: torch.Tensor, future_data: torch.Tensor, batch_seen: int, epoch: int, train: bool, **kwargs) -> torch.Tensor:
        """feedforward function of STGCN.

        Args:
            history_data (torch.Tensor): historical data with shape [B, L, N, C]

        Returns:
            torch.Tensor: prediction with shape [B, L, N, C]
        """
        x = history_data.permute(0, 3, 1, 2).contiguous()

        x = self.st_blocks(x)
        x = self.output(x)

        x = x.transpose(2, 3)
        return x

#### AGCRN

In [ ]:
class AVWGCN(nn.Module):
    def __init__(self, dim_in, dim_out, cheb_k, embed_dim):
        super(AVWGCN, self).__init__()
        self.cheb_k = cheb_k
        self.weights_pool = nn.Parameter(
            torch.FloatTensor(embed_dim, cheb_k, dim_in, dim_out))
        self.bias_pool = nn.Parameter(torch.FloatTensor(embed_dim, dim_out))

    def forward(self, x, node_embeddings):
        # x shaped[B, N, C], node_embeddings shaped [N, D] -> supports shaped [N, N]
        # output shape [B, N, C]
        node_num = node_embeddings.shape[0]
        supports = F.softmax(
            F.relu(torch.mm(node_embeddings, node_embeddings.transpose(0, 1))), dim=1)
        support_set = [torch.eye(node_num).to(supports.device), supports]
        # default cheb_k = 3
        for k in range(2, self.cheb_k):
            support_set.append(torch.matmul(
                2 * supports, support_set[-1]) - support_set[-2])
        supports = torch.stack(support_set, dim=0)
        # N, cheb_k, dim_in, dim_out
        weights = torch.einsum(
            'nd,dkio->nkio', node_embeddings, self.weights_pool)
        bias = torch.matmul(node_embeddings, self.bias_pool)  # N, dim_out
        x_g = torch.einsum("knm,bmc->bknc", supports,
                           x)  # B, cheb_k, N, dim_in
        x_g = x_g.permute(0, 2, 1, 3)  # B, N, cheb_k, dim_in
        x_gconv = torch.einsum('bnki,nkio->bno', x_g,
                               weights) + bias  # b, N, dim_out
        return x_gconv

class AGCRNCell(nn.Module):
    def __init__(self, node_num, dim_in, dim_out, cheb_k, embed_dim):
        super(AGCRNCell, self).__init__()
        self.node_num = node_num
        self.hidden_dim = dim_out
        self.gate = AVWGCN(dim_in+self.hidden_dim, 2 *
                           dim_out, cheb_k, embed_dim)
        self.update = AVWGCN(dim_in+self.hidden_dim,
                             dim_out, cheb_k, embed_dim)

    def forward(self, x, state, node_embeddings):
        # x: B, num_nodes, input_dim
        # state: B, num_nodes, hidden_dim
        state = state.to(x.device)
        input_and_state = torch.cat((x, state), dim=-1)
        z_r = torch.sigmoid(self.gate(input_and_state, node_embeddings))
        z, r = torch.split(z_r, self.hidden_dim, dim=-1)
        candidate = torch.cat((x, z*state), dim=-1)
        hc = torch.tanh(self.update(candidate, node_embeddings))
        h = r*state + (1-r)*hc
        return h

    def init_hidden_state(self, batch_size):
        return torch.zeros(batch_size, self.node_num, self.hidden_dim)

class AVWDCRNN(nn.Module):
    def __init__(self, node_num, dim_in, dim_out, cheb_k, embed_dim, num_layers=1):
        super(AVWDCRNN, self).__init__()
        assert num_layers >= 1, 'At least one DCRNN layer in the Encoder.'
        self.node_num = node_num
        self.input_dim = dim_in
        self.num_layers = num_layers
        self.dcrnn_cells = nn.ModuleList()
        self.dcrnn_cells.append(
            AGCRNCell(node_num, dim_in, dim_out, cheb_k, embed_dim))
        for _ in range(1, num_layers):
            self.dcrnn_cells.append(
                AGCRNCell(node_num, dim_out, dim_out, cheb_k, embed_dim))

    def forward(self, x, init_state, node_embeddings):
        # shape of x: (B, T, N, D)
        # shape of init_state: (num_layers, B, N, hidden_dim)
        assert x.shape[2] == self.node_num and x.shape[3] == self.input_dim
        seq_length = x.shape[1]
        current_inputs = x
        output_hidden = []
        for i in range(self.num_layers):
            state = init_state[i]
            inner_states = []
            for t in range(seq_length):
                state = self.dcrnn_cells[i](
                    current_inputs[:, t, :, :], state, node_embeddings)
                inner_states.append(state)
            output_hidden.append(state)
            current_inputs = torch.stack(inner_states, dim=1)
        # current_inputs: the outputs of last layer: (B, T, N, hidden_dim)
        # output_hidden: the last state for each layer: (num_layers, B, N, hidden_dim)
        #last_state: (B, N, hidden_dim)
        return current_inputs, output_hidden

    def init_hidden(self, batch_size):
        init_states = []
        for i in range(self.num_layers):
            init_states.append(
                self.dcrnn_cells[i].init_hidden_state(batch_size))
        # (num_layers, B, N, hidden_dim)
        return torch.stack(init_states, dim=0)

class AGCRN(nn.Module):
    """
    Paper: Adaptive Graph Convolutional Recurrent Network for Trafﬁc Forecasting
    Official Code: https://github.com/LeiBAI/AGCRN
    Link: https://arxiv.org/abs/2007.02842
    Venue: NeurIPS 2020
    Task: Spatial-Temporal Forecasting
    """

    def __init__(self, num_nodes, input_dim, rnn_units, output_dim, horizon, num_layers, default_graph, embed_dim, cheb_k):
        super(AGCRN, self).__init__()
        self.num_node = num_nodes
        self.input_dim = input_dim
        self.hidden_dim = rnn_units
        self.output_dim = output_dim
        self.horizon = horizon
        self.num_layers = num_layers

        self.default_graph = default_graph
        self.node_embeddings = nn.Parameter(torch.randn(
            self.num_node, embed_dim), requires_grad=True)

        self.encoder = AVWDCRNN(num_nodes, input_dim, rnn_units, cheb_k,
                                embed_dim, num_layers)

        # predictor
        self.end_conv = nn.Conv2d(
            1, horizon * self.output_dim, kernel_size=(1, self.hidden_dim), bias=True)

        self.init_param()

    def init_param(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
            else:
                nn.init.uniform_(p)

    def forward(self, history_data: torch.Tensor, future_data: torch.Tensor, batch_seen: int, epoch: int, train: bool, **kwargs) -> torch.Tensor:
        """Feedforward function of AGCRN.

        Args:
            history_data (torch.Tensor): inputs with shape [B, L, N, C].

        Returns:
            torch.Tensor: outputs with shape [B, L, N, C]
        """

        init_state = self.encoder.init_hidden(history_data.shape[0])
        output, _ = self.encoder(
            history_data, init_state, self.node_embeddings)  # B, T, N, hidden
        output = output[:, -1:, :, :]  # B, 1, N, hidden

        # CNN based predictor
        output = self.end_conv((output))  # B, T*C, N, 1
        output = output.squeeze(-1).reshape(-1, self.horizon,
                                            self.output_dim, self.num_node)
        output = output.permute(0, 1, 3, 2)  # B, T, N, C

        return output


#### Metrics

In [ ]:
def compute_metrics(output, target):
    """Вычисление метрик MAE, RMSE, MAPE."""
    abs_error = torch.abs(output - target).sum().item()
    mae = abs_error / len(target)
    rmse = ((output - target) ** 2).sum().item() / len(target)
    rmse = rmse ** 0.5
    mape = (abs_error / torch.abs(target).sum().item()) if torch.abs(target).sum().item() != 0 else 0
    return mae, rmse, mape

def train_val_test_model(model, train_loader, val_loader, test_loader, epochs, writer):
    best_val_loss = float('inf')

    for epoch in range(epochs):
        # === Тренировка ===
        model.train()
        train_loss, train_mae, train_rmse, train_mape = 0.0, 0.0, 0.0, 0.0
        train_loader_tqdm = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} - Training", leave=False)

        for step, (x, y) in enumerate(train_loader_tqdm):
            # x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            output = model(x).squeeze(-1)

            loss = criterion(output, y)
            loss.backward()

            optimizer.step()
            train_loss += loss.item() * x.size(0)

            mae, rmse, mape = compute_metrics(output, y)
            train_mae += mae
            train_rmse += rmse
            train_mape += mape

            # Запись метрик после каждого шага
            writer.add_scalar("Loss/Train", loss.item(), epoch * len(train_loader) + step)
            writer.add_scalar("MAE/Train", mae, epoch * len(train_loader) + step)
            writer.add_scalar("RMSE/Train", rmse, epoch * len(train_loader) + step)
            writer.add_scalar("MAPE/Train", mape, epoch * len(train_loader) + step)

        train_loss /= len(train_loader.dataset)
        train_mae /= len(train_loader)
        train_rmse /= len(train_loader)
        train_mape /= len(train_loader)

        # Запись метрик в конце эпохи
        writer.add_scalar("Loss/Train_Avg", train_loss, epoch + 1)
        writer.add_scalar("MAE/Train_Avg", train_mae, epoch + 1)
        writer.add_scalar("RMSE/Train_Avg", train_rmse, epoch + 1)
        writer.add_scalar("MAPE/Train_Avg", train_mape, epoch + 1)

        # === Валидация ===
        model.eval()
        val_loss, val_mae, val_rmse, val_mape = 0.0, 0.0, 0.0, 0.0
        val_loader_tqdm = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{epochs} - Validation", leave=False)

        with torch.no_grad():
            for step, (x, y) in enumerate(val_loader_tqdm):
                # x, y = x.to(device), y.to(device)
                output = model(x).squeeze(-1)
                loss = criterion(output, y)
                val_loss += loss.item() * x.size(0)

                mae, rmse, mape = compute_metrics(output, y)
                val_mae += mae
                val_rmse += rmse
                val_mape += mape

                # Запись метрик после каждого шага
                writer.add_scalar("Loss/Validation", loss.item(), epoch * len(val_loader) + step)
                writer.add_scalar("MAE/Validation", mae, epoch * len(val_loader) + step)
                writer.add_scalar("RMSE/Validation", rmse, epoch * len(val_loader) + step)
                writer.add_scalar("MAPE/Validation", mape, epoch * len(val_loader) + step)

        val_loss /= len(val_loader.dataset)
        val_mae /= len(val_loader)
        val_rmse /= len(val_loader)
        val_mape /= len(val_loader)

        # Запись метрик в конце эпохи
        writer.add_scalar("Loss/Validation_Avg", val_loss, epoch + 1)
        writer.add_scalar("MAE/Validation_Avg", val_mae, epoch + 1)
        writer.add_scalar("RMSE/Validation_Avg", val_rmse, epoch + 1)
        writer.add_scalar("MAPE/Validation_Avg", val_mape, epoch + 1)

        # === Тестирование ===
        test_loss, test_mae, test_rmse, test_mape = 0.0, 0.0, 0.0, 0.0
        test_loader_tqdm = tqdm(test_loader, desc=f"Epoch {epoch + 1}/{epochs} - Testing", leave=False)

        with torch.no_grad():
            for step, (x, y) in enumerate(test_loader_tqdm):
                # x, y = x.to(device), y.to(device)
                output = model(x).squeeze(-1)
                loss = criterion(output, y)
                test_loss += loss.item() * x.size(0)

                mae, rmse, mape = compute_metrics(output, y)
                test_mae += mae
                test_rmse += rmse
                test_mape += mape

                # Запись метрик после каждого шага
                writer.add_scalar("Loss/Test", loss.item(), epoch * len(test_loader) + step)
                writer.add_scalar("MAE/Test", mae, epoch * len(test_loader) + step)
                writer.add_scalar("RMSE/Test", rmse, epoch * len(test_loader) + step)
                writer.add_scalar("MAPE/Test", mape, epoch * len(test_loader) + step)

        test_loss /= len(test_loader.dataset)
        test_mae /= len(test_loader)
        test_rmse /= len(test_loader)
        test_mape /= len(test_loader)

        # Запись метрик в конце эпохи
        writer.add_scalar("Loss/Test_Avg", test_loss, epoch + 1)
        writer.add_scalar("MAE/Test_Avg", test_mae, epoch + 1)
        writer.add_scalar("RMSE/Test_Avg", test_rmse, epoch + 1)
        writer.add_scalar("MAPE/Test_Avg", test_mape, epoch + 1)

        # Сохранение лучшей модели
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), f"{writer.log_dir}best_model.pth")

    writer.close()


#### Train

In [ ]:
# === Устройство ===
device = torch.device("cuda") if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# === Гиперпараметры ===
num_nodes = train_data.shape[1]

# === Функция для инициализации модели ===
def create_gwnet_model(use_lags=False, use_filter=False, use_time_emb=False, use_dynamic_adj=False):
    emb_dim = 4 if use_time_emb else 0
    return GWNet(num_nodes=num_nodes,
                 in_dim=1,
                 emb_dim=emb_dim,
                 use_lags=use_lags,
                 use_filter=use_filter,
                 use_time_emb=use_time_emb,
                 use_dynamic_adj=use_dynamic_adj).to(device)

# === Перебор всех комбинаций флагов ===
# flags = ['use_lags', 'use_filter', 'use_time_emb', 'use_dynamic_adj']
# combinations = list(itertools.product([False, True], repeat=4))

flags = ['use_filter']
combinations = list(itertools.product([True, True], repeat=1))

for combo in combinations:
    config = dict(zip(flags, combo))
    print(f"\n=== Запуск конфигурации: {config} ===")

    # Создание модели
    model = create_gwnet_model(**config)

    # Оптимизатор и функция потерь
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    # TensorBoard логирование
    log_dir = "runs 2/GWNet_Logs only filter/PEMS03/" + "_".join([f"{k}={int(v)}" for k, v in config.items()])
    writer = SummaryWriter(log_dir=log_dir)

    # Тестовый вывод параметров модели
    train_iter = iter(train_loader)
    history_data = next(train_iter)[0].to(device)
    summary(model, input_data=history_data)

    # Обучение и валидация
    train_val_test_model(model, train_loader, val_loader, test_loader,
                         epochs=50, writer=writer)
